In [1]:
from imports import *  # noqa: F403

I0000 00:00:1783364716.506169    4808 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
required = ["training.csv", "test.csv"]
data_dir = "./data"

os.makedirs(data_dir, exist_ok=True)

missing = [f for f in required if not os.path.isfile(os.path.join(data_dir, f))]

if missing:
    print(f"Missing files: {missing}. Downloading...")
    !kaggle competitions download -c DontGetKicked -p {data_dir}

    while any(f.endswith(".zip") for f in os.listdir(data_dir)):
        for item in os.listdir(data_dir):
            if item.endswith(".zip"):
                zip_path = os.path.join(data_dir, item)
                with zipfile.ZipFile(zip_path, "r") as z:
                    z.extractall(data_dir)
                os.remove(zip_path)
                print(f"Extracted and removed: {item}")

    print("Done.")
else:
    print("All files already exist.")

All files already exist.


In [3]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

pd.set_option("display.max_colwidth", 500)
pd.set_option("display.width", 1000)

### Task 1

Download the data from the [Don'tGetKicked contest](https://www.kaggle.com/c/DontGetKicked).

Design a train/validation/test split.

Use the "PurchDate" field for the split, test must be later in time than validation, same goes for validation and train: train.PurchDate < valid.PurchDate < test.PurchDate.
Use the first 33% of the dates for the train, the last 33% of the dates for the test, and the middle 33% for the validation set. *Don't use the test dataset until the end!*

Use sklearn's LabelEncoder or OneHotEncoder to preprocess categorical variables. Be careful with data leakage (fit Encoder to train and apply to validation & test). Consider another coding approach if you encounter new categorical values in validation & test (not seen in training).

In [ ]:
from data_pipeline import prepare_data

X_train, X_val, X_test, y_train, y_val, y_test = prepare_data()

### Task 2 - 3

Create a Python class, MLP with 1 hidden layer and sigmoid activation function at the end of the network.
It should support *fit, predict_proba* and *predict methods*. Also, the number of neurons in the hidden layer and the activation function must be parameters of your class.

- Initialize the network with small random numbers.
- Use Log Loss (Binary Cross Entropy) as the loss function.
- Implement a forward pass; you can use a fixed batch size like 32, forward pass maps arrays of shape (batch_size, number_of_features) to arrays of shape (batch_size, 2) where 2 means dimensions of [probability_of_0, probability_of_1] output space.
- Write down the gradients of the loss function with respect to the parameters of the net.
- Use gradients to perform Backprop. Implement Backprop.
- Implement SGD algorithm to tune model parameters.
- Design a basic train-validation loop: iterate over the training dataset, batch by batch, update the parameters of the network, and check the quality of the model using the validation set.
- Write code to update network weights using SGD or Adam.

With your MLP module and careful network engineering, you must obtain at least a 0.15 Gini score on the validation dataset. You can train for more than 1 epoch, use different activation functions, and different optimizers (like SGD or Adam).

In [6]:
# def objective(trial):
#     l2_lambda = trial.suggest_float("l2_lambda", 0.001, 0.75, log=False)
#     model = MLP(
#         l2_lambda=l2_lambda,
#     )

#     model.fit(
#         X_train,
#         y_train,
#         X_val,
#         y_val,
#         epochs=100,
#         batch_size=128,
#         verbose=False,
#     )

#     proba_train = model.predict_proba(X_train)[:, 1]
#     proba_val = model.predict_proba(X_val)[:, 1]

#     train_gini = 2 * roc_auc_score(y_train, proba_train) - 1
#     val_gini = 2 * roc_auc_score(y_val, proba_val) - 1

#     penalty = abs(train_gini - val_gini)
#     score = val_gini - 0.5 * penalty

#     trial.set_user_attr("train_gini", train_gini)
#     trial.set_user_attr("val_gini", val_gini)
#     trial.set_user_attr("penalty", penalty)
#     trial.set_user_attr("l2_lambda", l2_lambda)

#     return score


# study = optuna.create_study(direction="maximize", pruner=MedianPruner())
# study.optimize(objective, n_trials=30, show_progress_bar=True)

# print("Best trial:")
# print("  Params:", study.best_params)
# print("  Score:", study.best_value)
# print("  Train Gini:", study.best_trial.user_attrs["train_gini"])
# print("  Val Gini:", study.best_trial.user_attrs["val_gini"])
# print("  Penalty:", study.best_trial.user_attrs["penalty"])
# print("  L2 lambda:", study.best_trial.user_attrs["l2_lambda"])

In [7]:
results = {}

In [8]:
params_custom_MLP = {
    "activation_func": "sigmoid",
    "optimizer": "adam",
    "l2_lambda": 0.03438351351362118,
}

model_custom_MLP = MLP(**params_custom_MLP)
model_custom_MLP.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    early_stopping_min_delta=0.001,
    early_stopping_patience=10,
    batch_size=128,
    verbose=True,
)
proba = model_custom_MLP.predict_proba(X_train)[:, 1]
train_gini = 2 * roc_auc_score(y_train, proba) - 1

proba = model_custom_MLP.predict_proba(X_val)[:, 1]
val_gini = 2 * roc_auc_score(y_val, proba) - 1
print(f"{train_gini} - {val_gini}")

results["model_custom_MLP"] = val_gini

Epoch   1 | Loss: 0.4443 | Val Gini: 0.4258
Epoch   2 | Loss: 0.3532 | Val Gini: 0.3967
Epoch   3 | Loss: 0.3517 | Val Gini: 0.3851
Epoch   4 | Loss: 0.3508 | Val Gini: 0.3958
Epoch   5 | Loss: 0.3527 | Val Gini: 0.3915
Epoch   6 | Loss: 0.3506 | Val Gini: 0.3886
Epoch   7 | Loss: 0.3472 | Val Gini: 0.4012
Epoch   8 | Loss: 0.3544 | Val Gini: 0.3769
Epoch   9 | Loss: 0.3550 | Val Gini: 0.4413
Epoch  10 | Loss: 0.3496 | Val Gini: 0.4090
Epoch  11 | Loss: 0.3494 | Val Gini: 0.3827
Epoch  12 | Loss: 0.3487 | Val Gini: 0.3944
Epoch  13 | Loss: 0.3496 | Val Gini: 0.4242
Epoch  14 | Loss: 0.3534 | Val Gini: 0.4071
Epoch  15 | Loss: 0.3529 | Val Gini: 0.3859
Epoch  16 | Loss: 0.3513 | Val Gini: 0.4303
Epoch  17 | Loss: 0.3516 | Val Gini: 0.4230
Epoch  18 | Loss: 0.3503 | Val Gini: 0.4159
Epoch  19 | Loss: 0.3522 | Val Gini: 0.4210
Early stopping at epoch 19
0.45650665335265894 - 0.4413180995039401


### Task 4

Use sklearn's MLPClassifier and check its performance on the validation dataset. Is it better than your module? If so, why?

In [9]:
params_sk_MLP = {
    "hidden_layer_sizes": 100,
    "activation": "logistic",
    "solver": "adam",
    "alpha": 0.03438351351362118,
    "early_stopping": True,
    "random_state": 42,
}

model_sk = skMLP(**params_sk_MLP)
model_sk.fit(X_train, y_train)


proba = model_sk.predict_proba(X_train)[:, 1]
gini_train = 2 * roc_auc_score(y_train, proba) - 1

proba = model_sk.predict_proba(X_val)[:, 1]
gini_val = 2 * roc_auc_score(y_val, proba) - 1
print(f"{gini_train} - {gini_val}")

results["model_sklearn_MLP"] = gini_val

0.5454064429615963 - 0.4423708018840764


### Task 5

Implement and try different activation functions: sigmoid, ReLU, cosine. Remember that you have to derive gradients for each different activation function. Which activation function gives the best performance on the validation dataset?

In [10]:
for act_func in ["relu", "cos", "tanh", "elu"]:
    params = {
        "learning_rate": 0.01,
        "optimizer": "adam",
        "l2_lambda": 0.03438351351362118,
    }
    model = MLP(**params)
    model.fit(
        X_train,
        y_train,
        X_val,
        y_val,
        early_stopping_min_delta=0.001,
        early_stopping_patience=10,
        batch_size=128,
        verbose=True,
    )

    proba = model.predict_proba(X_val)[:, 1]
    gini = 2 * roc_auc_score(y_val, proba) - 1
    print(f"Функция активации: {act_func}, Gini : {gini}")
    results[f"model_{act_func}"] = gini

Epoch   1 | Loss: 0.4443 | Val Gini: 0.4258
Epoch   2 | Loss: 0.3532 | Val Gini: 0.3967
Epoch   3 | Loss: 0.3517 | Val Gini: 0.3851
Epoch   4 | Loss: 0.3508 | Val Gini: 0.3958
Epoch   5 | Loss: 0.3527 | Val Gini: 0.3915
Epoch   6 | Loss: 0.3506 | Val Gini: 0.3886
Epoch   7 | Loss: 0.3472 | Val Gini: 0.4012
Epoch   8 | Loss: 0.3544 | Val Gini: 0.3769
Epoch   9 | Loss: 0.3550 | Val Gini: 0.4413
Epoch  10 | Loss: 0.3496 | Val Gini: 0.4090
Epoch  11 | Loss: 0.3494 | Val Gini: 0.3827
Epoch  12 | Loss: 0.3487 | Val Gini: 0.3944
Epoch  13 | Loss: 0.3496 | Val Gini: 0.4242
Epoch  14 | Loss: 0.3534 | Val Gini: 0.4071
Epoch  15 | Loss: 0.3529 | Val Gini: 0.3859
Epoch  16 | Loss: 0.3513 | Val Gini: 0.4303
Epoch  17 | Loss: 0.3516 | Val Gini: 0.4230
Epoch  18 | Loss: 0.3503 | Val Gini: 0.4159
Epoch  19 | Loss: 0.3522 | Val Gini: 0.4210
Early stopping at epoch 19
Функция активации: relu, Gini : 0.4413180995039401
Epoch   1 | Loss: 0.4443 | Val Gini: 0.4258
Epoch   2 | Loss: 0.3532 | Val Gini: 0.396

### Task 6

Design an MLP module with 1 hidden layer using any high level deep learning framework: Pytorch, Keras, or Tensorflow. Check its performance on the validation dataset.  Is it better than your module? If so, why?

In [11]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

best_init = keras.initializers.GlorotUniform(seed=SEED)
best_lr = 0.0001
best_neurons = 50

inputs = keras.Input(shape=(X_train.shape[1],))
x = keras.layers.Dense(
    best_neurons,
    activation="sigmoid",
    kernel_initializer=best_init,
    bias_initializer=keras.initializers.Zeros(),
)(inputs)
outputs = keras.layers.Dense(
    1,
    activation="sigmoid",
    kernel_initializer=keras.initializers.GlorotUniform(seed=SEED + 1),
    bias_initializer=keras.initializers.Zeros(),
)(x)
model_tf = keras.Model(inputs=inputs, outputs=outputs)

model_tf.compile(
    optimizer=keras.optimizers.Adam(learning_rate=best_lr),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

history = model_tf.fit(
    X_train,
    y_train,
    batch_size=128,
    epochs=100,
    validation_data=(X_val, y_val),
    shuffle=True,
    callbacks=[early_stop],
    verbose=0,
)

proba_tf = model_tf.predict(X_val).ravel()
gini_tf = 2 * roc_auc_score(y_val, proba_tf) - 1
print(f"Keras (best params) Gini: {gini_tf:.4f}")

results["model_tensorflow_best"] = gini_tf

W0000 00:00:1783364768.608381    4808 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Epoch 29: early stopping
Restoring model weights from the end of the best epoch: 24.
761/761 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Keras (best params) Gini: 0.4306


In [12]:
df_results = pd.DataFrame(results, index=["Метрика Gini на валидации"])
display(df_results.T)

,Метрика Gini на валидации
model_custom_MLP,0.441318
model_sklearn_MLP,0.442371
model_relu,0.441318
model_cos,0.441318
model_tanh,0.441318
model_elu,0.441318
model_tensorflow_best,0.430634


In [13]:
proba = model_sk.predict_proba(X_test)[:, 1]
gini = 2 * roc_auc_score(y_test, proba) - 1
print(f"Gini лучшей модели на тестовой выборке: {gini}")

Gini лучшей модели на тестовой выборке: 0.4679426646751743
